In [1]:
# TASK 1 - Design a Class

class TrainingRun:
    
    run_count = 0

    def __init__(self, model_name, learning_rate):
        if not self.validate_learning_rate(learning_rate):
            raise ValueError("Learning rate must be between 0 and 1.")
            
        self.model_name = model_name
        self.learning_rate = learning_rate
        self.status = "Not Started"

        TrainingRun.run_count += 1

    def start(self):
        """Start the training run."""
        self.status = "Running"
        print(f"Training started for {self.model_name}")

    def summary(self):
        """Display the current training run status."""
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

    @classmethod
    def from_config(cls, config_dict):
        """Create a TrainingRun object from a configuration dictionary."""
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(learning_rate):
        """Validate that learning rate is within a sane range."""
        return 0 < learning_rate < 1

run1 = TrainingRun("ResNet", 0.01)
run2 = TrainingRun("LSTM", 0.001)
run3 = TrainingRun("Transformer", 0.0001)

print("Run count:", TrainingRun.run_count)

print("\nRun 1:")
run1.summary()

print("\nRun 2:")
run2.summary()

print("\nRun 3:")
run3.summary()

print("\nStarting Run 1:")
run1.start()
run1.summary()

config = {
    "model_name": "CNN",
    "learning_rate": 0.005
}

run4 = TrainingRun.from_config(config)

print("\nRun 4 created using from_config:")
run4.summary()

print("\nFinal run count:", TrainingRun.run_count)

Run count: 3

Run 1:
Model: ResNet, Learning Rate: 0.01, Status: Not Started

Run 2:
Model: LSTM, Learning Rate: 0.001, Status: Not Started

Run 3:
Model: Transformer, Learning Rate: 0.0001, Status: Not Started

Starting Run 1:
Training started for ResNet
Model: ResNet, Learning Rate: 0.01, Status: Running

Run 4 created using from_config:
Model: CNN, Learning Rate: 0.005, Status: Not Started

Final run count: 4


In [2]:
# TASK 2 - Encapsulation

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        self.__api_key = "fake-api-key-12345"

        TrainingRun.run_count += 1

    @property
    def status(self):
        """Getter for the protected _status attribute."""
        return self._status

    @status.setter
    def status(self, value):
        """Setter that validates the status value."""
        if value not in ("pending", "running", "done"):
            raise ValueError(
                "Status must be 'pending', 'running', or 'done'."
            )

        self._status = value

    def start(self):
        self.status = "running"
        print(f"Training started for {self.model_name}")

    def complete(self):
        self.status = "done"
        print(f"Training completed for {self.model_name}")

    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

run = TrainingRun("ResNet", 0.01)

print("Initial status:", run.status)

run.status = "running"
print("Updated status:", run.status)

run.complete()
print("Final status:", run.status)

# This will raise AttributeError because __api_key
# is name-mangled inside the class.

try:
    print(run.__api_key)
except AttributeError:
    print("Direct access to __api_key is not allowed.")


print("Access using name mangling:", run._TrainingRun__api_key)

try:
    run.status = "paused"
except ValueError as e:
    print("Error:", e)

Initial status: pending
Updated status: running
Training completed for ResNet
Final status: done
Direct access to __api_key is not allowed.
Access using name mangling: fake-api-key-12345
Error: Status must be 'pending', 'running', or 'done'.


In [3]:
# TASK 3 - Inheritance

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        self.__api_key = "fake-api-key-12345"

        TrainingRun.run_count += 1

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        if value not in ("pending", "running", "done"):
            raise ValueError(
                "Status must be 'pending', 'running', or 'done'."
            )
        self._status = value

    def start(self):
        self.status = "running"
        print(f"Training started for {self.model_name}")

    def complete(self):
        self.status = "done"
        print(f"Training completed for {self.model_name}")

    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

class LRSchedulerRun(TrainingRun):

    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule

    def summary(self):
        super().summary()
        print(f"Learning Rate Schedule: {self.schedule}")

        if self.schedule:
            print(f"Current Scheduled Learning Rate: {self.schedule[0]}")


run1 = TrainingRun("ResNet", 0.01)

print("TrainingRun:")
run1.summary()

run2 = LRSchedulerRun(
    "LSTM",
    0.001,
    [0.001, 0.0005, 0.0001, 0.00005]
)

print("\nLRSchedulerRun:")
run2.summary()

print("\nTotal Training Runs:", TrainingRun.run_count)
print("LRSchedulerRun count:", LRSchedulerRun.run_count)

TrainingRun:
Model: ResNet, Learning Rate: 0.01, Status: pending

LRSchedulerRun:
Model: LSTM, Learning Rate: 0.001, Status: pending
Learning Rate Schedule: [0.001, 0.0005, 0.0001, 5e-05]
Current Scheduled Learning Rate: 0.001

Total Training Runs: 2
LRSchedulerRun count: 2


In [4]:
# TASK 4 - polymorphism

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        self.__api_key = "fake-api-key-12345"

        TrainingRun.run_count += 1

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        if value not in ("pending", "running", "done"):
            raise ValueError(
                "Status must be 'pending', 'running', or 'done'."
            )
        self._status = value

    def start(self):
        self.status = "running"
        print(f"Training started for {self.model_name}")

    def complete(self):
        self.status = "done"
        print(f"Training completed for {self.model_name}")

    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

class LRSchedulerRun(TrainingRun):

    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule

    def summary(self):
        # Reuse the parent summary
        super().summary()

        print(f"Learning Rate Schedule: {self.schedule}")

        if self.schedule:
            print(
                f"Current Scheduled Learning Rate: "
                f"{self.schedule[0]}"
            )


class EarlyStoppingRun(TrainingRun):

    def __init__(self, model_name, learning_rate, patience):
        super().__init__(model_name, learning_rate)
        self.patience = patience

    def summary(self):
        # Different implementation of summary()
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}, "
            f"Patience: {self.patience} epochs"
        )


def print_all_summaries(runs):
    for run in runs:
        run.summary()

run1 = TrainingRun(
    "ResNet",
    0.01
)

run2 = LRSchedulerRun(
    "LSTM",
    0.001,
    [0.001, 0.0005, 0.0001]
)

run3 = EarlyStoppingRun(
    "Transformer",
    0.0001,
    5
)

runs = [run1, run2, run3]


print("Training Run Summaries:")
print("-----------------------")

print_all_summaries(runs)
print("\nTotal Training Runs:", TrainingRun.run_count)

Training Run Summaries:
-----------------------
Model: ResNet, Learning Rate: 0.01, Status: pending
Model: LSTM, Learning Rate: 0.001, Status: pending
Learning Rate Schedule: [0.001, 0.0005, 0.0001]
Current Scheduled Learning Rate: 0.001
Model: Transformer, Learning Rate: 0.0001, Status: pending, Patience: 5 epochs

Total Training Runs: 3


In [5]:
# TASK 5 - Duck Typing vs Interfaces

from abc import ABC, abstractmethod

# PART 1: DUCK TYPING

class DuckCleaner:
    def process(self, data):
        return data.strip()


class DuckTokenizer:
    def process(self, data):
        return data.split()


class DuckNormalizer:
    def process(self, data):
        return [word.lower() for word in data]


def run_pipeline_duck(steps, data):
    for step in steps:
        data = step.process(data)

    return data

duck_steps = [
    DuckCleaner(),
    DuckTokenizer(),
    DuckNormalizer()
]

data = "  HELLO PYTHON WORLD  "

duck_result = run_pipeline_duck(duck_steps, data)

print("=== Duck Typing ===")
print(duck_result)


# PART 2: ABSTRACT BASE CLASS

class Step(ABC):

    @abstractmethod
    def process(self, data):
        pass


class Cleaner(Step):
    def process(self, data):
        return data.strip()


class Tokenizer(Step):
    def process(self, data):
        return data.split()


class Normalizer(Step):
    def process(self, data):
        return [word.lower() for word in data]


def run_pipeline(steps, data):
    for step in steps:
        data = step.process(data)

    return data


steps = [
    Cleaner(),
    Tokenizer(),
    Normalizer()
]

abc_result = run_pipeline(steps, data)

print("\n=== ABC Version ===")
print(abc_result)


# PART 3: ABSTRACT CLASS INSTANTIATION
try:
    step = Step()
except TypeError as e:
    print("\n=== ABC Instantiation Test ===")
    print("Step() cannot be instantiated.")
    print("Error:", e)


# PART 4: TEAM DISCUSSION

# Duck typing is useful when we want flexible code and the objects
# only need to support the required method.
#
# An ABC is useful on a team when we want to explicitly define a
# contract and ensure every subclass implements required methods.

=== Duck Typing ===
['hello', 'python', 'world']

=== ABC Version ===
['hello', 'python', 'world']

=== ABC Instantiation Test ===
Step() cannot be instantiated.
Error: Can't instantiate abstract class Step without an implementation for abstract method 'process'


In [6]:
# TASK 6 - Dunder Methods

class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"

        TrainingRun.run_count += 1

    def __str__(self):
        """
        Human-readable representation of the object.
        Used by print() and str().
        """
        return (
            f"TrainingRun(model='{self.model_name}', "
            f"lr={self.learning_rate})"
        )

    def __eq__(self, other):
        """
        Two TrainingRun objects are equal when they have
        the same model name and learning rate.
        """
        if not isinstance(other, TrainingRun):
            return NotImplemented

        return (
            self.model_name == other.model_name
            and self.learning_rate == other.learning_rate
        )

    def __repr__(self):
        """
        Developer-oriented representation of the object.
        __str__ is mainly for users, while __repr__ is mainly
        for debugging and development.
        """
        return (
            f"TrainingRun("
            f"model_name='{self.model_name}', "
            f"learning_rate={self.learning_rate})"
        )

run1 = TrainingRun("gpt-mini", 0.01)
run2 = TrainingRun("gpt-mini", 0.01)
run3 = TrainingRun("LSTM", 0.001)


print("Using __str__:")
print(run1)

print("\nUsing __eq__:")

print("run1 == run2:", run1 == run2)
print("run1 == run3:", run1 == run3)


print("\nUsing __repr__:")
print(repr(run1))

print("\nObject:")
print(run3)

Using __str__:
TrainingRun(model='gpt-mini', lr=0.01)

Using __eq__:
run1 == run2: True
run1 == run3: False

Using __repr__:
TrainingRun(model_name='gpt-mini', learning_rate=0.01)

Object:
TrainingRun(model='LSTM', lr=0.001)
